Import Functions

In [0]:
from pyspark.sql.functions import col, count, avg, length, desc, when

In [0]:
df = spark.table("log_catalog.silver.comments_silver")

In [0]:
spark.sql("USE CATALOG log_catalog")
spark.sql("USE SCHEMA gold")

DataFrame[]

**Comment Count per Post**

Grouped data by postId and calculated total comments per post.

In [0]:
comment_count = df.groupBy("postId") \
    .agg(count("*").alias("comment_count"))

In [0]:
comment_count.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.comment_count")

**Trending Posts**

Sorted posts based on highest number of comments.
This identifies trending or most engaging posts.

In [0]:
trending_posts = comment_count.orderBy(desc("comment_count"))

In [0]:
trending_posts.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.trending_posts")

**User activity**

Useful for analyzing user engagement.


In [0]:
user_activity = df.groupBy("user") \
    .agg(count("*").alias("total_comments"))

In [0]:
user_activity.write.format("delta").mode("overwrite").saveAsTable("log_catalog.gold.user_activity")

**Top active users**

Sorted users based on highest activity

Helps identify most active users.

In [0]:
top_users = user_activity.orderBy(desc("total_comments"))

In [0]:
top_users.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.top_users")

**engagement metrics**

Calculated total comments and average comment length per post.



In [0]:
engagement = df.groupBy("postId") \
    .agg(count("*").alias("comment_count"),avg(length(col("comment"))).alias("avg_comment_length"))

In [0]:
engagement.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.engagement_metrics")

**Avg comments per posts**

Calculated overall average number of comments per post.

This gives a benchmark for platform engagement

In [0]:
avg_comments = comment_count.agg(
    avg("comment_count").alias("avg_comments_per_post"))

In [0]:
avg_comments.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.avg_comments")

**User segmentations**

Created a new column to classify users based on activity levels.

In [0]:
user_segmentation = user_activity.withColumn(
    "user_type",
    when(col("total_comments") >= 10, "High")
    .when(col("total_comments") >= 5, "Medium")
    .otherwise("Low"))

In [0]:
user_segmentation.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.user_segmentation")

**NULL CHECK**

Counted non-null values for each column in the dataset.


In [0]:
null_check = df.select([
    count(col(c)).alias(c) for c in df.columns])

In [0]:
null_check.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.null_check")

**DUPLICATE CHECK**

Saved duplicate check results to ADLS storage.

In [0]:
duplicate_check = df.groupBy(
    "postId", "commentId", "user", "comment"
).agg(count("*").alias("duplicate_count"))

In [0]:
duplicate_check.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://log-data@stlogprocessing123.dfs.core.windows.net/gold/duplicate_check")

**ERROR RATE CALCULATIONS**



In [0]:
from pyspark.sql.functions import when

Added a column to flag comments containing errors.

In [0]:
df_with_error = df.withColumn( "is_error",when(col("comment").contains("error"), 1).otherwise(0))

Calculated total logs and number of error logs.

This helps measure system error frequency.


In [0]:
error_rate = df_with_error.agg(count("*").alias("total_logs"),
count(when(col("is_error") == 1, True)).alias("error_logs"))

Computed error rate as a ratio of error logs to total logs.



In [0]:
error_rate = error_rate.withColumn("error_rate",col("error_logs") / col("total_logs"))

Saved error rate data as a Delta table.

In [0]:
error_rate.write.format("delta").mode("overwrite") \
    .saveAsTable("log_catalog.gold.error_rate")

In [0]:
error_rate.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://log-data@stlogprocessing123.dfs.core.windows.net/gold/error_rate")

Show Tables

In [0]:
spark.sql("SHOW TABLES IN log_catalog.gold").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
|    gold|      avg_comments|      false|
|    gold|     comment_count|      false|
|    gold|engagement_metrics|      false|
|    gold|        error_rate|      false|
|    gold|        null_check|      false|
|    gold|         top_users|      false|
|    gold|    trending_posts|      false|
|    gold|     user_activity|      false|
|    gold| user_segmentation|      false|
+--------+------------------+-----------+

